# Day 4 — Solution: Adjusted Prices, Simple vs Log

## Setup

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    adj = get_prices("KO", start="2012-01-01", field="adj_close")
    raw = get_prices("KO", start="2012-01-01", field="raw_close")
else:
    rng = np.random.default_rng(11)
    n, p0, yld = 2500, 100.0, 0.03
    r_price = rng.normal(0.0002, 0.010, n)
    idx = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=n)
    raw = pd.DataFrame({"KO": p0 * np.cumprod(1 + r_price)}, index=idx)
    adj = pd.DataFrame({"KO": (p0 * np.cumprod(1 + r_price + yld / 252))}, index=idx)
n_years = (adj.index[-1] - adj.index[0]).days / 365.25
print(f"sample: {n_years:.1f} years")

## Part A — the dividend gap

In [ ]:
def annualized_return(price_series, n_years):
    """CAGR: compound the whole sample, take the 1/n_years root."""
    growth = price_series.iloc[-1] / price_series.iloc[0]
    return growth ** (1 / n_years) - 1

total_cagr = annualized_return(adj["KO"], n_years)
price_cagr = annualized_return(raw["KO"], n_years)
print(f"total-return CAGR: {total_cagr:.2%}")
print(f"price-only   CAGR: {price_cagr:.2%}")
print(f"gap: {(total_cagr - price_cagr) * 100:.2f} pp per year")

Real KO (2012→now) runs ~2–3 pp/year of dividend gap; the synthetic 3%
yielder gives ~3 pp by construction. **A1:** the gap is dividends — the
adjusted series reinvests them, the raw one drops them on the floor. A study
using `raw_close` understates the stock's return by (roughly) its dividend
yield, *every year, compounding*. Over a decade that's a third of your money
for a mature payer — and it silently corrupts any cross-sectional ranking
that mixes payers and non-payers (value strategies do exactly that).

In [ ]:
g_adj = adj["KO"] / adj["KO"].iloc[0]
g_raw = raw["KO"] / raw["KO"].iloc[0]
plt.plot(g_adj.index, g_adj, label="total return (adj)")
plt.plot(g_raw.index, g_raw, label="price only (raw)")
plt.legend(); plt.title("Growth of $1: the dividend wedge")
plt.show()
print(f"end-of-sample ratio of growth factors: {g_adj.iloc[-1] / g_raw.iloc[-1]:.2f}x")

**A2:** the $1 paths diverge multiplicatively — after ~10 years at ~3% yield
the total-return dollar is roughly 1.3× the price-only dollar.

## Part B — simple vs log

In [ ]:
r = adj["KO"].pct_change().dropna()      # simple
lr = np.log1p(r)                          # log: ln(1 + r)

**B1 — additivity.**

In [ ]:
weekly_log = lr.resample("W-FRI").sum().dropna()
weekly_price = adj["KO"].resample("W-FRI").last().pct_change().dropna()
check = np.log1p(weekly_price)
print((weekly_log - check).abs().max())

The max difference is ~0 (floating point): the *sum of daily log returns* in
a week equals the *log of the week's growth factor*. That's the additivity
property — the reason log returns are the natural currency of time-series
statistics.

**B2 — where they diverge.**

In [ ]:
grid = np.linspace(-0.2, 0.2, 401)
plt.plot(grid, grid, label="45° line (r = lr)")
plt.plot(grid, np.log1p(grid), label="log return")
plt.xlabel("simple return r"); plt.ylabel("log return")
plt.legend(); plt.title("ln(1+r) vs r")
plt.show()

rel_err = (np.log1p(r) - r).abs() / r.abs()
ok = rel_err[abs(r) > 1e-4] < 0.10
print(f"share of days within 10% relative error: {ok.mean():.1%}")
print(f"largest |r| with <10% relative error: ~{abs(r[rel_err < 0.10]).max():.2%}")

They agree to <10% relative error for |r| up to roughly ±20%; the divergence
grows with r² (ln(1+r) ≈ r − r²/2 + …). Daily stock returns are usually well
inside that band — which is why "log ≈ simple" is safe *until the day it
isn't*.

**B3 — crash asymmetry.**

In [ ]:
for s in [-0.10, -0.50, -0.90]:
    print(f"simple {s:+.0%} -> log {np.log1p(s):+.4f}")

−10% → −0.105; −50% → −0.693; −90% → −2.303. Log returns are unbounded
below while simple returns floor at −100% — the *price* can't go negative,
so in log space a crash is just a large negative number (this is exactly why
we do return statistics in logs and portfolio arithmetic in levels). "Log
returns are symmetric" is a modeling assumption you will test against real
data in module 03 — and it will fail, mildly but expensively.

## Part C — the two-panel figure

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(adj.index, adj["KO"]); ax[0].set_title("adjusted price")
ax[1].plot(r.index, r); ax[1].set_title("daily returns")
best_day, worst_day = r.idxmax(), r.idxmin()
ax[1].annotate(f"best {r.max():+.1%}\n{best_day.date()}", xy=(best_day, r.max()),
               xytext=(10, 10), textcoords="offset points")
ax[1].annotate(f"worst {r.min():+.1%}\n{worst_day.date()}", xy=(worst_day, r.min()),
               xytext=(10, -25), textcoords="offset points")
plt.tight_layout(); plt.show()

## Common mistakes

- `np.log(r)` (log of a return) instead of `np.log1p(r)` (log of the growth
  factor). The first crashes on negative returns — a useful crash. On small
  positive r it *appears* to work, which is worse.
- Comparing CAGRs of one adjusted series and one raw series — you've measured
  dividends, not performance.
- Averaging daily simple returns and calling it an annual return (that's an
  arithmetic mean of a compounding process; module 01.3 shows why the
  geometric mean is the honest one).